[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ersilia-os/ub-cedd-projects-workshop/blob/main/projects/blue/notebooks/blue_pharmacophore.ipynb)

# Deriving a pharmacophore for the CpABC1 pocket

**Blue group · Cryptosporidiosis**

Silymarin binds CpABC1, but weakly. To find better molecules we need a description of what the
pocket wants, rather than of silymarin itself. This notebook derives that description, a
**pharmacophore**, and saves it as a query you can run against a library of purchasable compounds.

## What you will do

- Load the CpABC1–silymarin complex and check the protein is read correctly
- Derive a pharmacophore for the pocket with PharmacoNet
- Compare the features with the contacts silymarin actually makes
- Choose which features to search on, and view them in the pocket
- Save the query as a Pharmit session file

## Setup

Run the cell below first. In Colab it downloads the workshop repository (including the data) and installs the packages this project needs. It takes about a minute. **Don't change it.**

In [ ]:
PROJECT = "blue"
NEEDS_GPU = False
import os, sys, shutil, subprocess
if "google.colab" in sys.modules:
    repo_dir = "/content/ub-cedd-projects-workshop"
    if not os.path.exists(repo_dir):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ersilia-os/ub-cedd-projects-workshop.git", repo_dir], check=True)
    else:
        subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only"], check=True)
    os.chdir(f"{repo_dir}/projects/{PROJECT}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
for _cached in [m for m in sys.modules if m == "scripts" or m.startswith("scripts.")]:
    del sys.modules[_cached]  # forget helper modules imported before the pull above
has_gpu = shutil.which("nvidia-smi") is not None and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
print(f"Python {sys.version.split()[0]} | GPU: {'yes' if has_gpu else 'no'} | Folder: {os.getcwd()}")
if NEEDS_GPU and not has_gpu:
    print("WARNING: this notebook needs a GPU. Go to Runtime > Change runtime type, choose CPU, and run this cell again.")

## 1. What a pharmacophore is

A **pharmacophore** is a list of what a molecule needs in order to bind, and where. Each item is a
**feature**: a point in space plus the kind of chemistry expected there, such as "a hydrogen bond
donor here" or "something greasy there". A molecule matches if it can place the right kind of atom
at each point.

We use **PharmacoNet** ([Seo and Kim, Chemical Science 2024](https://doi.org/10.1039/D4SC04854G)),
a model that reads the **pocket**, not the ligand. Silybin, the active molecule in silymarin, is
used for one thing only: the average of its atom positions, which tells PharmacoNet where in the
protein to look. Nothing about its chemistry goes in. That is what makes the comparison in
section 4 a real test.

PharmacoNet needs Python 3.10 to 3.12. This cell says whether the runtime has it.

> **Note:** if this fails in Colab, go to *Runtime > Change runtime type*, set the runtime version
> to **2026.07**, then *Runtime > Disconnect and delete runtime* and run the notebook again.

In [ ]:
import platform, sys

version = sys.version_info[:2]
print(f"Python {platform.python_version()}")
if not (3, 10) <= version < (3, 13):
    print("PharmacoNet needs Python 3.10 to 3.12. See the note above.")

## 2. The CpABC1 target

Two files describe the target, both in `data/`:

- `cpabc1_receptor.pdb` is the protein on its own: 11,435 heavy atoms, 1,431 residues, chain A.
- `silymarin_ligand.sdf` is silybin, 35 heavy atoms, in the same coordinate frame.

Both were split out of the group's `cpabc1_silymarin.pdb` complex, so the ligand sits exactly where
it was docked.

In [ ]:
from pathlib import Path

DATA = Path("data")
RECEPTOR = DATA / "cpabc1_receptor.pdb"
LIGAND = DATA / "silymarin_ligand.sdf"

print(f"{RECEPTOR}: {RECEPTOR.stat().st_size / 1e6:.1f} MB")
print(f"{LIGAND}: {LIGAND.stat().st_size / 1e3:.1f} kB")

PharmacoNet reads only the 20 standard amino acid names and silently drops any residue named
anything else, with no warning. Schrödinger Maestro, which the group used to prepare the complex,
writes `HIE`, `HID` and `HIP` for histidine and `CYX` for a cysteine in a disulphide bond. Those
would disappear without trace, so we check the names first.

In [ ]:
STANDARD = set("ALA ARG ASN ASP CYS GLN GLU GLY HIS ILE LEU LYS MET PHE PRO SER THR TRP TYR VAL".split())

atoms = [line for line in open(RECEPTOR) if line.startswith(("ATOM", "HETATM"))]
odd = {line[17:20].strip() for line in atoms} - STANDARD

print(f"{len(atoms)} atoms, non-standard residue names: {sorted(odd) if odd else 'none'}")

## 3. Derive the pharmacophore

PharmacoNet's weights are a 139 MB file that is not in the repository, so we download it the first
time and keep it in `data/downloads/`, which git ignores. The authors host it on a single Google
Drive link that is often rate-limited, so this tries the workshop's own copy first.

The download takes a minute or two.

In [ ]:
import urllib.error, urllib.request

MIRROR = ("https://github.com/ersilia-os/ub-cedd-projects-workshop/"
          "releases/download/pmnet-weights-v2.2.0/pharmaconet_weights.tar")
DRIVE = "https://drive.google.com/uc?id=1gzjdM7bD3jPm23LBcDXtkSk18nETL04p"
WEIGHTS = DATA / "downloads" / "pharmaconet_weights.tar"
WEIGHTS.parent.mkdir(parents=True, exist_ok=True)

if not WEIGHTS.exists():
    try:
        urllib.request.urlretrieve(MIRROR, WEIGHTS)
    except urllib.error.HTTPError:
        import gdown
        gdown.download(DRIVE, str(WEIGHTS), quiet=False)

print(f"{WEIGHTS.stat().st_size / 1e6:.0f} MB")

Now run the model. `score_threshold` decides how many features come back.

The score is a **percentile, not a probability**: 0.9 means a spot scored higher than 90 per cent of
those seen in training. PharmacoNet's default of 0.85 returned ten features here, nearly all greasy
ones, with no hydrogen bond acceptors at all. At 0.5 the polar features appear, and those are what
make the search selective later.

The model itself takes about five minutes on a laptop, and a little longer on Colab.

> **Note:** CpABC1 is a membrane transporter, while PharmacoNet was trained on soluble proteins.
> This target is outside what the model has seen, so treat the result as a hypothesis to test.

In [ ]:
import time
from pmnet.module import PharmacoNet

start = time.time()
module = PharmacoNet(device="cpu", weight_path=str(WEIGHTS), score_threshold=0.5, verbose=False)
model = module.run(str(RECEPTOR), ref_ligand_path=str(LIGAND))

print(f"{len(model.nodes)} features in {time.time() - start:.0f} s")

Each feature says what a **ligand** atom there would have to be (`type`), where it should sit
(`center`), and how far off it may be (`radius`).

In [ ]:
import pandas as pd

features = pd.DataFrame([{"type": node.type, "score": round(float(node.score), 3),
                          "radius": round(float(node.radius), 2),
                          "x": node.center[0], "y": node.center[1], "z": node.center[2]}
                         for node in model.nodes])

print(features["type"].value_counts().to_string())
features.sort_values("score", ascending=False).head(10)

## 4. Compare with silybin's contacts

PharmacoNet never saw silybin's atoms, so we can now ask a fair question: do the features it found
from the protein alone land where silybin actually sits?

Measured straight from the complex, silybin's seven closest polar contacts are **Ser862 (2.73 Å),
Asn858 (2.77), Ser888 (2.77), Thr353 (2.80), Tyr961 (2.87), Gln1101 (2.90)** and **Arg965 (3.05)**.

Expect partial agreement. The model describes the whole pocket, so it also finds spots silybin
never touches, and those are exactly where a better molecule could gain grip. A miss is not a
failure.

In [ ]:
import numpy as np
from rdkit import Chem

ligand = Chem.MolFromMolFile(str(LIGAND))
positions = ligand.GetConformer().GetPositions()

protein = [line for line in open(RECEPTOR) if line.startswith("ATOM")]
coords = np.array([[float(l[30:38]), float(l[38:46]), float(l[46:54])] for l in protein])
residues = [f"{l[17:20].strip()}{int(l[22:26])}" for l in protein]

features["to_silybin"] = [round(float(np.linalg.norm(positions - np.asarray(n.center), axis=1).min()), 2)
                          for n in model.nodes]
features["residue"] = [residues[int(np.linalg.norm(coords - np.asarray(n.hotspot_position), axis=1).argmin())]
                       for n in model.nodes]

How much of silybin's binding did the model recover on its own?

In [ ]:
CONTACTS = {"SER862", "ASN858", "SER888", "THR353", "TYR961", "GLN1101", "ARG965"}

on_pose = int((features.to_silybin <= 3.0).sum())
found = sorted({r for r in features.residue if r in CONTACTS})

print(f"{on_pose} of {len(features)} features sit within 3.0 A of silybin")
print(f"{len(found)} of silybin's 7 polar contacts recovered: {found}")

## 5. Translate to Pharmit's language

[Pharmit](https://pharmit.csb.pitt.edu) is a free web service that searches libraries of
purchasable compounds for molecules matching a pharmacophore. It defines exactly six feature types,
one fewer than PharmacoNet, so halogen features have nowhere to go and are dropped.

The rest map straight across, because both describe the feature from the **ligand's** side.

In [ ]:
TO_PHARMIT = {"Hydrophobic": "Hydrophobic", "Aromatic": "Aromatic",
              "Cation": "PositiveIon", "Anion": "NegativeIon",
              "HBond_donor": "HydrogenDonor", "HBond_acceptor": "HydrogenAcceptor",
              "Halogen": None}

# Pharmit's own colours, from its js/pharmit.js, so the picture below matches the website
PHARMIT_COLOURS = {"Aromatic": "purple", "HydrogenDonor": "0xf0f0f0",
                   "HydrogenAcceptor": "orange", "Hydrophobic": "green",
                   "NegativeIon": "red", "PositiveIon": "blue"}

dropped = [n.type for n in model.nodes if not TO_PHARMIT[n.type]]
print(f"{len(model.nodes)} features, {len(dropped)} dropped ({sorted(set(dropped))})")

## 6. Choose which features to search on

This step decides whether the search returns nothing, something, or everything. Pharmit requires a
hit to match **every** feature that is switched on, so each one you add makes the search stricter.

Three rules, each learned from a search that went wrong:

- **Stay on the pose.** Only features within 3 Å of silybin may be switched on. The best-scoring
  greasy feature here sits 7.7 Å away in a neighbouring cavity, and switching it on drags the search
  out of the binding site.
- **Variety before quantity.** Take the best feature of each kind before taking a second of any
  kind. A query made mostly of greasy points matches almost anything.
- **Tighten if there are too many hits.** `RADIUS_SCALE` shrinks every tolerance. At 1.0 this query
  returned far too many compounds, so we use 0.7.

In [ ]:
def choose(nodes, ligand_positions, enable=6, on_pose=3.0):
    """Return the features to search on: on the pose, varied, best-scoring first."""
    usable = sorted((n for n in nodes if TO_PHARMIT[n.type]), key=lambda n: -float(n.score))
    near = [n for n in usable
            if np.linalg.norm(ligand_positions - np.asarray(n.center), axis=1).min() <= on_pose]

    picked, seen = [], set()
    for node in near:
        if TO_PHARMIT[node.type] not in seen:
            seen.add(TO_PHARMIT[node.type])
            picked.append(id(node))
    for node in near:
        if len(picked) >= enable:
            break
        if id(node) not in picked:
            picked.append(id(node))
    return usable, set(picked[:enable])

Every usable feature goes into the file, but only the chosen ones are switched on. You can
toggle the others in the browser without coming back here.

In [ ]:
usable, enabled = choose(model.nodes, positions, enable=6)

chosen = pd.DataFrame([{"type": TO_PHARMIT[n.type], "score": round(float(n.score), 3),
                        "to_silybin": features.to_silybin[i]}
                       for i, n in enumerate(model.nodes) if id(n) in enabled])
print(f"{len(usable)} usable features, {len(enabled)} switched on")
chosen

## 7. Build the session file

The protein is written into the file as well, which is what lets Pharmit draw the pocket around the
query. It also makes the file about 1 MB.

In [ ]:
RADIUS_SCALE = 0.7

session = {"points": [], "exselect": "receptor", "extolerance": 1, "max-hits": 25000,
           "receptor": RECEPTOR.read_text(), "recname": "cpabc1_receptor.pdb"}

for node in usable:
    x, y, z = (float(v) for v in node.center)
    session["points"].append({
        "name": TO_PHARMIT[node.type], "has_vec": False,
        "x": round(x, 3), "y": round(y, 3), "z": round(z, 3),
        "radius": round(float(node.radius) * RADIUS_SCALE, 2),
        "enabled": id(node) in enabled, "vector_on": 0,
        "svector": {"x": 1, "y": 0, "z": 0},
        "minsize": "", "maxsize": "", "selected": False})

query = pd.DataFrame([p for p in session["points"] if p["enabled"]])
query[["name", "x", "y", "z", "radius"]]

## 8. See the query in the pocket

Solid spheres are the features the search will use, wireframe ones are loaded but switched off.
Silybin is in green sticks for reference. Drag to rotate, scroll to zoom.

> **Exercise:** find the greasy feature sitting away from silybin, outside the pocket. That is the
> one that would ruin the search if it were switched on.

In [ ]:
import py3Dmol

view = py3Dmol.view(width="100%", height=520)
view.addModel(session["receptor"], "pdb")
view.setStyle({"cartoon": {"color": "lightgrey", "opacity": 0.55}})
view.addModel(LIGAND.read_text(), "sdf")
view.setStyle({"model": 1}, {"stick": {"colorscheme": "greenCarbon", "radius": 0.15}})

for point in session["points"]:
    view.addSphere({"center": {"x": point["x"], "y": point["y"], "z": point["z"]},
                    "radius": point["radius"], "color": PHARMIT_COLOURS[point["name"]],
                    "opacity": 0.85 if point["enabled"] else 0.25,
                    "wireframe": not point["enabled"]})

view.zoomTo({"model": 1})
view.show()

## 9. Save the query

The file is written to `outputs/`, which is not part of the repository. In Colab it also downloads
to your computer, because the Colab runtime is wiped when it disconnects.

Take it to [pharmit.csb.pitt.edu/search.html](https://pharmit.csb.pitt.edu/search.html), choose
**Load Session**, pick a library such as MolPort, and run the search. Save the results as SDF files
and upload them to the group's Drive folder: the next notebook turns them into a compound list.

> **Note:** the chosen features include a `NegativeIon` point, so every hit will carry an acid
> group. Silybin has none, so this is the one place the query asks for chemistry the reference
> molecule does not have. To look for silymarin-like hits instead, drop that feature type before
> section 6 and let a second greasy point take its place.

In [ ]:
import json

OUT = Path("outputs")
OUT.mkdir(exist_ok=True)
out = OUT / "cpabc1_pharmit_session.json"
out.write_text(json.dumps(session, indent=2))

print(f"{out}: {out.stat().st_size / 1e6:.2f} MB, {len(query)} features switched on")
if "google.colab" in sys.modules:
    from google.colab import files
    files.download(str(out))

## Summary

- PharmacoNet read the CpABC1 pocket and proposed a pharmacophore, using silybin only to say where
  to look.
- Some of its features landed on silybin's own contacts, and others sit elsewhere in the pocket,
  where a better molecule could bind more tightly.
- Six features, all on the pose and of varied kinds, became a Pharmit query saved as
  `cpabc1_pharmit_session.json`.

**Next:** run the query on Pharmit, then open `blue_pharmit_hits.ipynb` to turn the results into a
list of compounds you could order.